In [1]:
import os
import cv2
import glob
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import wandb
from tqdm import tqdm
import torchvision.transforms.functional as F
from torchvision.utils import save_image

# Initialize wandb
wandb.init(
    project="ablation-veo-real",
    name="Epochs-10-BatchSize-32-LR-0.001-Adam-ResNet50 (400x400 Center Crop)",
    config={
        "epochs": 10,
        "batch_size": 32,
        "lr": 0.001,
        "optimizer": "Adam",
        "architecture": "resnet50",
        "img_size": 400,
        "split": [0.7, 0.15, 0.15],
        "weight_decay": 1e-4,
        "max_batches_per_epoch": 100,
        "max_val_batches_per_epoch": 50,
        "max_test_batches": 100
    }
)
config = wandb.config

# Device
device = torch.device('cuda:3' if torch.cuda.is_available() else 'cpu')

# Custom Dataset for Video Frames
class VideoFrameDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        frame, label = self.samples[idx]
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image = Image.fromarray(image)
        if self.transform:
            image = self.transform(image)
        return image, label

# Load and split video frames into train/val/test
def load_and_split_video_frames(video_dir, transform, split=(0.7, 0.15, 0.15), frame_skip=1):
    class_map = {'veo3': 0, 'real': 1}
    all_samples = []

    for class_name in ['veo3', 'real']:
        label = class_map[class_name]
        class_path = os.path.join(video_dir, class_name)
        video_paths = glob.glob(os.path.join(class_path, "*.mp4"))

        for video_path in video_paths:
            cap = cv2.VideoCapture(video_path)
            frame_idx = 0
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break
                if frame_idx % frame_skip == 0:
                    all_samples.append((frame.copy(), label))
                frame_idx += 1
            cap.release()

    print(f"Total frames collected: {len(all_samples)}")
    random.shuffle(all_samples)
    total = len(all_samples)
    train_end = int(split[0] * total)
    val_end = train_end + int(split[1] * total)

    train_samples = all_samples[:train_end]
    val_samples = all_samples[train_end:val_end]
    test_samples = all_samples[val_end:]

    return (
        VideoFrameDataset(train_samples, transform),
        VideoFrameDataset(val_samples, transform),
        VideoFrameDataset(test_samples, transform),
        class_map
    )

# Transforms
transform = transforms.Compose([
    transforms.CenterCrop(config.img_size),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load video dataset
train_ds, val_ds, test_ds, class_to_idx = load_and_split_video_frames(
    video_dir='AI_Video_Generation_Dataset/real-veo',
    transform=transform,
    split=config.split,
    frame_skip=5
)

#save an iamge from the dataset to verify
sample_dir = "sample_crops_400"
if not os.path.exists(sample_dir):
    os.makedirs(sample_dir)

# Helper: unnormalize a tensor image
def unnormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return tensor * std + mean

# Save up to N samples per class
num_to_save = 5
saved = {"real": 0, "veo3": 0}
max_save = {"real": num_to_save, "veo3": num_to_save}

for i in range(len(train_ds)):
    image_tensor, label = train_ds[i]
    label_name = "real" if label == 1 else "veo3"
    
    if saved[label_name] < max_save[label_name]:
        unnorm_img = unnormalize(image_tensor)
        save_path = os.path.join(sample_dir, f"{label_name}_{saved[label_name]}.png")
        save_image(unnorm_img, save_path)
        saved[label_name] += 1

    if all(saved[l] >= max_save[l] for l in saved):
        break



# DataLoaders
train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=config.batch_size, shuffle=False)

# Model
model = models.resnet50(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)

# Training loop
for epoch in range(config.epochs):
    model.train()
    train_loss, train_correct, total = 0.0, 0, 0
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.epochs} [Training]", leave=False)

    for batch_idx, (inputs, labels) in enumerate(train_loop):
        if batch_idx >= config.max_batches_per_epoch:
            break

        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        train_correct += (preds == labels).sum().item()
        total += labels.size(0)
        train_loop.set_postfix(loss=loss.item())

    train_acc = 100.0 * train_correct / total
    avg_train_loss = train_loss / (batch_idx + 1)

    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/{config.epochs} [Validation]", leave=False)

    with torch.no_grad():
        for val_batch_idx, (inputs, labels) in enumerate(val_loop):
            if val_batch_idx >= config.max_val_batches_per_epoch:
                break

            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
            val_loop.set_postfix(loss=loss.item())

    val_acc = 100.0 * val_correct / val_total
    avg_val_loss = val_loss / (val_batch_idx + 1)

    # Log to wandb
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "train_accuracy": train_acc,
        "val_loss": avg_val_loss,
        "val_accuracy": val_acc
    })

    print(f"Epoch {epoch+1}: Train Acc: {train_acc:.2f}%, Val Acc: {val_acc:.2f}%")

# Save model
torch.save(model.state_dict(), "resnet_video_frame_model.pth")
wandb.save("resnet_video_frame_model.pth")

# Final test evaluation
def evaluate_on_test(model, test_loader):
    model.eval()
    test_correct, test_total, test_loss = 0, 0, 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch_idx, (inputs, labels) in enumerate(test_loader):
            if batch_idx >= config.max_test_batches:
                break
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            test_correct += (preds == labels).sum().item()
            test_total += labels.size(0)

    test_acc = 100.0 * test_correct / test_total
    avg_test_loss = test_loss / (batch_idx + 1)

    print(f"\nFinal Test Accuracy: {test_acc:.2f}%, Test Loss: {avg_test_loss:.4f}")
    wandb.log({"test_accuracy": test_acc, "test_loss": avg_test_loss})

    wandb.log({
        "confusion_matrix": wandb.plot.confusion_matrix(
            probs=None,
            y_true=all_labels,
            preds=all_preds,
            class_names=list(class_to_idx.keys())
        )
    })

evaluate_on_test(model, test_loader)



wandb: Currently logged in as: aarushishukla (aarushishukla-uc-santa-barbara) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Total frames collected: 4487


/home/aarushi/Detecting-Deepfakes/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/aarushi/Detecting-Deepfakes/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1: Train Acc: 90.48%, Val Acc: 84.40%


Epoch 2: Train Acc: 98.22%, Val Acc: 99.41%


...

Epoch 6: Train Acc: 99.30%, Val Acc: 98.81%


Epoch 7: Train Acc: 99.62%, Val Acc: 99.85%


Epoch 8: Train Acc: 99.46%, Val Acc: 99.85%


Epoch 9: Train Acc: 99.81%, Val Acc: 100.00%


Epoch 10: Train Acc: 99.75%, Val Acc: 100.00%

Final Test Accuracy: 100.00%, Test Loss: 0.0038
